# 26 · Visão geral da base (exploratória)

Primeira leitura da base da Vitaliza, sem gráficos: quantas contas existem, o perfil delas,
quem está ativo no corte (30/09/2026), por onde assinam e como usam o app.

Usa só as tabelas limpas de `dados/processados/`. Nenhuma decisão nova de limpeza.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")

P = Path("../dados/processados")
S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")

usuarios = pd.read_parquet(P / "usuarios_limpa.parquet")
assin = pd.read_parquet(P / "assinaturas_limpa.parquet")
trials = pd.read_parquet(P / "trials.parquet")
ativ = pd.read_parquet(P / "atividades_limpa.parquet")
rede = pd.read_parquet(P / "rede_por_conta.parquet")
posts = pd.read_parquet(P / "feed_posts_limpa.parquet")

for nome, d in [("usuarios", usuarios), ("assinaturas", assin), ("trials", trials),
                ("atividades", ativ), ("rede_por_conta", rede), ("feed_posts", posts)]:
    print(f"{nome:15s} {len(d):>9,} linhas | {d['user_id'].nunique():>6,} contas")

def dist(s, nome="contas"):
    t = s.value_counts(dropna=False).rename(nome).to_frame()
    t["%"] = t[nome] / t[nome].sum() * 100
    return t

def cruz(d, linha, coluna):
    n = pd.crosstab(d[linha], d[coluna], margins=True, margins_name="total")
    p = pd.crosstab(d[linha], d[coluna], normalize="columns", margins=True, margins_name="total") * 100
    return n, p

usuarios           63,929 linhas | 63,929 contas
assinaturas        32,583 linhas | 32,583 contas
trials             46,996 linhas | 46,996 contas
atividades      1,252,488 linhas | 36,551 contas
rede_por_conta     63,929 linhas | 63,929 contas
feed_posts        119,187 linhas | 26,878 contas


## 1. Quantas contas e em que situação

In [2]:
base = usuarios.merge(assin[["user_id", "plano_atual", "canal_cobranca", "inicio", "status_atual",
                             "renovacao_automatica", "data_fim", "motivo_fim", "ativa_no_corte",
                             "duracao_dias", "inicio_antes_do_log"]],
                      on="user_id", how="left", validate="one_to_one")
print("contas:", len(usuarios), "-> depois da junção com assinaturas:", len(base))
print("assinaturas sem conta em usuarios_limpa:", (~assin.user_id.isin(usuarios.user_id)).sum())

base = base.merge(trials[["user_id", "desfecho"]].rename(columns={"desfecho": "desfecho_trial"}),
                  on="user_id", how="left", validate="one_to_one")
print("depois da junção com trials:", len(base))
print("trials sem conta em usuarios_limpa:", (~trials.user_id.isin(usuarios.user_id)).sum())

def situacao(r):
    if r["ativa_no_corte"] is True: return "1. assinante ativo"
    if pd.notna(r["status_atual"]): return "2. ex-assinante"
    if r["desfecho_trial"] == "em_andamento": return "3. em teste grátis"
    if r["desfecho_trial"] == "expirou": return "4. teste expirou, não assinou"
    return "5. nunca iniciou teste"
base["situacao"] = base.apply(situacao, axis=1)
base["tem_assinatura"] = base["status_atual"].notna()

print()
print(dist(base["situacao"]).sort_index())

contas:

 63929 -> depois da junção com assinaturas: 63929
assinaturas sem conta em usuarios_limpa: 0
depois da junção com trials: 63929
trials sem conta em usuarios_limpa: 0



                               contas    %
situacao                                  
1. assinante ativo              14753 23.1
2. ex-assinante                 17830 27.9
3. em teste grátis                470  0.7
4. teste expirou, não assinou   19143 29.9
5. nunca iniciou teste          11733 18.4


In [3]:
# O arquivo quase não tem contas grátis anteriores a 07/2024 (decisoes.md): separar as duas eras.
n, p = cruz(base.assign(era=base.cadastro_antes_do_log.map({True: "cadastro antes de 07/2024", False: "cadastro desde 07/2024"})),
            "situacao", "era")
print(n); print(); print(p)

era                            cadastro antes de 07/2024  cadastro desde 07/2024  total
situacao                                                                               
1. assinante ativo                                  1462                   13291  14753
2. ex-assinante                                     3738                   14092  17830
3. em teste grátis                                     0                     470    470
4. teste expirou, não assinou                          0                   19143  19143
5. nunca iniciou teste                                 0                   11733  11733
total                                               5200                   58729  63929

era                            cadastro antes de 07/2024  cadastro desde 07/2024  total
situacao                                                                               
1. assinante ativo                                  28.1                    22.6   23.1
2. ex-assinante                

In [4]:
base["ano_cadastro"] = base.data_cadastro.dt.year
t = base.groupby("ano_cadastro").agg(contas=("user_id", "size"),
                                      ja_assinou=("tem_assinatura", "sum"),
                                      ativa_no_corte=("ativa_no_corte", lambda s: (s == True).sum()))
t["% já assinou"] = t.ja_assinou / t.contas * 100
t["% ativa no corte"] = t.ativa_no_corte / t.contas * 100
print(t)

              contas  ja_assinou  ativa_no_corte  % já assinou  % ativa no corte
ano_cadastro                                                                    
2020             436         436             200         100.0              45.9
2021            1358        1358             464         100.0              34.2
2022            1314        1314             320         100.0              24.4
2023            1423        1423             329         100.0              23.1
2024           11831        5976            1408          50.5              11.9
2025           25770       12247            4574          47.5              17.7
2026           21797        9829            7458          45.1              34.2


## 2. Perfil demográfico

Base toda, quem já assinou e assinantes ativos no corte, lado a lado (% da coluna).

In [5]:
base["faixa_idade"] = pd.cut(base.idade_no_corte.astype("float"), [0, 24, 34, 44, 54, 120],
                             labels=["até 24", "25-34", "35-44", "45-54", "55+"]).astype(object).fillna("não informado")
base["sexo_"] = base.sexo.fillna("não informado")
base["ativo_"] = base.ativa_no_corte == True

def perfil(col):
    grupos = {"todas as contas": base, "já assinou": base[base.tem_assinatura],
              "assinante ativo": base[base.ativo_], "ex-assinante": base[base.tem_assinatura & ~base.ativo_],
              "nunca assinou (desde 07/2024)": base[~base.tem_assinatura & ~base.cadastro_antes_do_log]}
    t = pd.DataFrame({k: v[col].value_counts(normalize=True) * 100 for k, v in grupos.items()})
    t.insert(0, "contas", base[col].value_counts())
    return t.sort_values("contas", ascending=False)

print("contas por grupo:", {"todas": len(base), "já assinou": int(base.tem_assinatura.sum()),
      "ativo": int(base.ativo_.sum()), "ex": int((base.tem_assinatura & ~base.ativo_).sum()),
      "nunca assinou desde 07/2024": int((~base.tem_assinatura & ~base.cadastro_antes_do_log).sum())})
perfis = {}
for col in ["faixa_idade", "sexo_", "uf", "cidade", "sistema", "canal_aquisicao", "relogio_conectado"]:
    perfis[col] = perfil(col)
    print(); print("==", col); print(perfis[col])

contas por grupo: {'todas': 63929, 'já assinou': 32583, 'ativo': 14753, 'ex': 17830, 'nunca assinou desde 07/2024': 31346}

== faixa_idade
               contas  todas as contas  já assinou  assinante ativo  ex-assinante  nunca assinou (desde 07/2024)
faixa_idade                                                                                                     
45-54           18950             29.6        29.7             29.5          29.8                           29.6
35-44           18906             29.6        29.2             29.5          28.9                           30.0
55+             15359             24.0        24.5             24.0          24.8                           23.6
25-34            6601             10.3        10.2             10.5          10.0                           10.4
não informado    3834              6.0         6.1              6.1           6.1                            5.9
até 24            279              0.4         0.4              0.5   


== sexo_
               contas  todas as contas  já assinou  assinante ativo  ex-assinante  nunca assinou (desde 07/2024)
sexo_                                                                                                           
F               31996             50.0        49.9             49.8          50.0                           50.2
M               28844             45.1        45.3             45.4          45.1                           45.0
não informado    3089              4.8         4.8              4.7           4.9                            4.8

== uf
    contas  todas as contas  já assinou  assinante ativo  ex-assinante  nunca assinou (desde 07/2024)
uf                                                                                                   
SP   31508             49.3        49.2             49.1          49.2                           49.4
RJ    7400             11.6        11.5             11.6          11.3                           11.7
MG    4462

== sistema
         contas  todas as contas  já assinou  assinante ativo  ex-assinante  nunca assinou (desde 07/2024)
sistema                                                                                                   
android   34596             54.1        52.5             50.7          54.0                           55.8
ios       29333             45.9        47.5             49.3          46.0                           44.2



== canal_aquisicao
                      contas  todas as contas  já assinou  assinante ativo  ex-assinante  nunca assinou (desde 07/2024)
canal_aquisicao                                                                                                        
organico               18256             28.6        30.9             31.4          30.4                           26.2
influenciador          13851             21.7        17.6             16.9          18.2                           25.9
meta_ads               11783             18.4        17.4             16.4          18.3                           19.5
indicacao               7993             12.5        14.2             15.1          13.5                           10.7
google_ads              6701             10.5        10.7             10.4          10.9                           10.3
parceria_corporativa    5345              8.4         9.2              9.8           8.8                            7.5

== relogio_conectad

In [6]:
i = base.idade_no_corte.dropna().astype(float)
print("idade no corte (todas):", i.describe()[["count", "mean", "25%", "50%", "75%", "min", "max"]].round(1).to_dict())
i = base.loc[base.ativo_, "idade_no_corte"].dropna().astype(float)
print("idade no corte (ativos):", i.describe()[["count", "mean", "25%", "50%", "75%", "min", "max"]].round(1).to_dict())
print()
print("sexo × faixa de idade, assinantes ativos (% do total informado)")
a = base[base.ativo_ & base.sexo.notna() & (base.faixa_idade != "não informado")]
print(len(a), "ativos com sexo e idade")
print(pd.crosstab(a.faixa_idade, a.sexo, normalize="all", margins=True) * 100)

pd.concat(perfis, names=["variavel", "valor"]).to_csv(S / "eda_visao_geral_perfil.csv", encoding="utf-8-sig")

idade no corte (todas): {'count': 60095.0, 'mean': 47.6, '25%': 39.0, '50%': 47.0, '75%': 55.0, 'min': 17.0, 'max': 74.0}
idade no corte (ativos): {'count': 13856.0, 'mean': 47.5, '25%': 39.0, '50%': 46.0, '75%': 55.0, 'min': 18.0, 'max': 74.0}

sexo × faixa de idade, assinantes ativos (% do total informado)
13197 ativos com sexo e idade
sexo           F    M   All
faixa_idade                
25-34        6.1  5.1  11.2
35-44       16.5 15.0  31.4
45-54       16.0 15.3  31.3
55+         13.5 12.2  25.6
até 24       0.3  0.2   0.5
All         52.3 47.7 100.0


## 3. Ativos e inativos no corte

Duas definições: **assinatura** (paga no corte) e **uso** (treinou nos últimos 30 dias, 01 a 30/09/2026).

In [7]:
print(dist(assin.status_atual, "assinaturas"))
print()
enc = assin[~assin.ativa_no_corte]
print("encerradas:", len(enc))
print(dist(enc.motivo_fim, "assinaturas"))
print()
at = assin[assin.ativa_no_corte]
print("ativas:", len(at))
print(dist(at.renovacao_automatica, "ativas"))
print()
print("tempo de casa das ativas no corte (dias):", at.duracao_dias.describe()[["mean", "25%", "50%", "75%", "max"]].round(0).to_dict())
print("duração das encerradas (dias):", enc.duracao_dias.describe()[["mean", "25%", "50%", "75%", "max"]].round(0).to_dict())
print()
print("assinaturas encerradas por ano do fim")
print(enc.data_fim.dt.year.value_counts().sort_index())

              assinaturas    %
status_atual                  
expirada            17758 54.5
ativa               14753 45.3
reembolsada            72  0.2

encerradas: 17830
                     assinaturas    %
motivo_fim                           
renovacao_desligada        16145 90.5
falha_pagamento             1321  7.4
chargeback                   292  1.6
reembolso                     72  0.4

ativas: 14753
                      ativas    %
renovacao_automatica             
sim                    12503 84.7
nao                     2250 15.3

tempo de casa das ativas no corte (dias): {'mean': 418.0, '25%': 121.0, '50%': 263.0, '75%': 554.0, 'max': 2213.0}
duração das encerradas (dias): {'mean': 375.0, '25%': 120.0, '50%': 256.0, '75%': 390.0, 'max': 2190.0}

assinaturas encerradas por ano do fim
data_fim
2024    1798
2025    7633
2026    8399
Name: count, dtype: int64


In [8]:
# Uso: último treino de cada conta em 2026 (atividades_limpa cobre 01/01 a 30/09/2026)
print("atividades:", len(ativ), "de", ativ.data.min().date(), "a", ativ.data.max().date())
ult = ativ.groupby("user_id").data.max().rename("ultimo_treino")
n30 = ativ[ativ.data > CORTE - pd.Timedelta(days=30)].groupby("user_id").size().rename("treinos_30d")
base = base.merge(ult, on="user_id", how="left").merge(n30, on="user_id", how="left")
base["treinos_30d"] = base.treinos_30d.fillna(0).astype(int)
dias = (CORTE - base.ultimo_treino).dt.days
base["uso"] = pd.cut(dias, [-1, 6, 29, 89, 400], labels=["a. treinou nos últimos 7 dias", "b. 8 a 30 dias", "c. 31 a 90 dias", "d. 91+ dias (em 2026)"]).astype(object)
base["uso"] = base.uso.fillna("e. nenhum treino em 2026")
print("contas depois das junções:", len(base))
n, p = cruz(base, "uso", "situacao")
print(n); print(); print(p)

atividades: 1252488 de 2026-01-01 a 2026-09-30


contas depois das junções:

 63929
situacao                       1. assinante ativo  2. ex-assinante  3. em teste grátis  4. teste expirou, não assinou  5. nunca iniciou teste  total
uso                                                                                                                                                 
a. treinou nos últimos 7 dias               10689              118                 331                            742                     523  12403
b. 8 a 30 dias                               2366              437                   0                           1340                     779   4922
c. 31 a 90 dias                               944             1503                   0                           1703                    1004   5154
d. 91+ dias (em 2026)                         578             5927                   0                           4728                    2839  14072
e. nenhum treino em 2026                      176             9845                 139             

In [9]:
a = base[base.ativo_]
print("assinantes ativos:", len(a))
print("treinaram nos últimos 30 dias:", (a.treinos_30d > 0).sum(), f"({(a.treinos_30d > 0).mean():.1%})")
print("pagam e não treinaram nos últimos 30 dias:", (a.treinos_30d == 0).sum(), f"({(a.treinos_30d == 0).mean():.1%})")
print()
print("treinos nos últimos 30 dias, entre ativos que treinaram:")
print(a.loc[a.treinos_30d > 0, "treinos_30d"].describe()[["mean", "25%", "50%", "75%", "max"]].round(1).to_dict())
print()
print("uso × plano entre os ativos (% da coluna)")
print(cruz(a, "uso", "plano_atual")[1])
print()
# Assinantes muito recentes ainda não tiveram 30 dias para treinar
a2 = a[a.inicio <= CORTE - pd.Timedelta(days=30)]
print("ativos com 30+ dias de assinatura:", len(a2), "| sem treino em 30 dias:", f"{(a2.treinos_30d == 0).mean():.1%}")
n = base[~base.tem_assinatura]
print("contas que nunca assinaram:", len(n), "| treinaram nos últimos 30 dias:", (n.treinos_30d > 0).sum(), f"({(n.treinos_30d > 0).mean():.1%})")
e = base[base.tem_assinatura & ~base.ativo_]
print("ex-assinantes:", len(e), "| treinaram nos últimos 30 dias:", (e.treinos_30d > 0).sum(), f"({(e.treinos_30d > 0).mean():.1%})")

assinantes ativos: 14753
treinaram nos últimos 30 dias: 13055 (88.5%)
pagam e não treinaram nos últimos 30 dias: 1698 (11.5%)

treinos nos últimos 30 dias, entre ativos que treinaram:
{'mean': 10.7, '25%': 6.0, '50%': 10.0, '75%': 14.0, 'max': 62.0}

uso × plano entre os ativos (% da coluna)
plano_atual                    anual  mensal  total
uso                                                
a. treinou nos últimos 7 dias   62.1    86.4   72.5
b. 8 a 30 dias                  18.3    12.9   16.0
c. 31 a 90 dias                 10.7     0.6    6.4
d. 91+ dias (em 2026)            6.8     0.0    3.9
e. nenhum treino em 2026         2.0     0.0    1.2

ativos com 30+ dias de assinatura: 13605 | sem treino em 30 dias: 12.4%
contas que nunca assinaram: 31346 | treinaram nos últimos 30 dias: 3715 (11.9%)
ex-assinantes: 17830 | treinaram nos últimos 30 dias: 555 (3.1%)


## 4. Por onde assinam: plano e canal de cobrança

In [10]:
assin["situacao"] = assin.ativa_no_corte.map({True: "ativa", False: "encerrada"})
print("todas as assinaturas:", len(assin))
print(dist(assin.canal_cobranca, "assinaturas")); print()
print(dist(assin.plano_atual, "assinaturas")); print()

t = assin.groupby(["canal_cobranca", "plano_atual"]).agg(assinaturas=("user_id", "size"), ativas=("ativa_no_corte", "sum"))
t["encerradas"] = t.assinaturas - t.ativas
t["% do total"] = t.assinaturas / t.assinaturas.sum() * 100
t["% das ativas"] = t.ativas / t.ativas.sum() * 100
t["% ainda ativa"] = t.ativas / t.assinaturas * 100
print(t); print()
for col in ["canal_cobranca", "plano_atual"]:
    g = assin.groupby(col).agg(assinaturas=("user_id", "size"), ativas=("ativa_no_corte", "sum"))
    g["% das ativas"] = g.ativas / g.ativas.sum() * 100
    g["% ainda ativa"] = g.ativas / g.assinaturas * 100
    print(g); print()
t.to_csv(S / "eda_visao_geral_plano_canal.csv", encoding="utf-8-sig")

# "% ainda ativa" mistura coortes: repetir só para quem começou desde 07/2024
r = assin[~assin.inicio_antes_do_log]
g = r.groupby(["canal_cobranca", "plano_atual"]).agg(assinaturas=("user_id", "size"), ativas=("ativa_no_corte", "sum"))
g["% ainda ativa"] = g.ativas / g.assinaturas * 100
print("só assinaturas iniciadas desde 07/2024:", len(r)); print(g)

todas as assinaturas: 32583
                assinaturas    %
canal_cobranca                  
app_store             11170 34.3
google_play           10948 33.6
web_cartao             7087 21.8
web_pix                3378 10.4

             assinaturas    %
plano_atual                  
mensal             19556 60.0
anual              13027 40.0

                            assinaturas  ativas  encerradas  % do total  % das ativas  % ainda ativa
canal_cobranca plano_atual                                                                          
app_store      anual               4552    3047        1505        14.0          20.7           66.9
               mensal              6618    2348        4270        20.3          15.9           35.5
google_play    anual               4529    2928        1601        13.9          19.8           64.7
               mensal              6419    2036        4383        19.7          13.8           31.7
web_cartao     anual               2859    180

In [11]:
b = base[base.tem_assinatura]
print("canal de cobrança × sistema do aparelho (% da linha), quem já assinou:", len(b))
print(pd.crosstab(b.sistema, b.canal_cobranca, normalize="index", margins=True) * 100)
print()
print("canal de cobrança × canal de aquisição (% da linha)")
print(pd.crosstab(b.canal_aquisicao, b.canal_cobranca, normalize="index", margins=True) * 100)
print()
print("plano × canal de aquisição (% da linha)")
print(pd.crosstab(b.canal_aquisicao, b.plano_atual, normalize="index", margins=True) * 100)
print()
print("teste grátis (desde 07/2024):", len(trials))
print(dist(trials.desfecho, "testes"))
g = trials[trials.desfecho != "em_andamento"].groupby("canal_trial").desfecho.agg(testes="size", converteu=lambda s: (s == "converteu").sum())
g["% converteu"] = g.converteu / g.testes * 100
print(g)

canal de cobrança × sistema do aparelho (% da linha), quem já assinou: 32583
canal_cobranca  app_store  google_play  web_cartao  web_pix
sistema                                                    
android               0.0         64.0        24.5     11.5
ios                  72.2          0.0        18.7      9.1
All                  34.3         33.6        21.8     10.4

canal de cobrança × canal de aquisição (% da linha)


canal_cobranca        app_store  google_play  web_cartao  web_pix
canal_aquisicao                                                  
google_ads                 32.8         34.4        22.6     10.2
indicacao                  34.2         33.7        22.4      9.7
influenciador              35.0         33.5        20.9     10.7
meta_ads                   34.2         33.3        22.5     10.0
organico                   34.5         33.5        21.2     10.8
parceria_corporativa       34.4         33.4        22.0     10.2
All                        34.3         33.6        21.8     10.4

plano × canal de aquisição (% da linha)
plano_atual           anual  mensal
canal_aquisicao                    
google_ads             41.3    58.7
indicacao              40.7    59.3
influenciador          30.7    69.3
meta_ads               42.0    58.0
organico               42.6    57.4
parceria_corporativa   42.3    57.7
All                    40.0    60.0

teste grátis (desde 07/2024): 46996
    

             testes  converteu  % converteu
canal_trial                                
app_store     15399       9458         61.4
google_play   16161       9140         56.6
web_cartao    10194       5939         58.3
web_pix        4772       2846         59.6


## 5. Como usam o app

### 5.1 Esportes (2026, até o corte)

In [12]:
print("treinos:", len(ativ), "| contas com treino em 2026:", ativ.user_id.nunique(), "de", len(base), f"({ativ.user_id.nunique() / len(base):.1%})")
ar_livre = {"corrida", "caminhada", "pedal"}
t = ativ.groupby("tipo").agg(treinos=("atividade_id", "size"), contas=("user_id", "nunique"),
                             duracao_mediana_min=("duracao_min", "median"))
t["% dos treinos"] = t.treinos / t.treinos.sum() * 100
t["% das contas que treinam"] = t.contas / ativ.user_id.nunique() * 100
t["treinos por conta"] = t.treinos / t.contas
t = t.sort_values("treinos", ascending=False)
print(t)
t.to_csv(S / "eda_visao_geral_esportes.csv", encoding="utf-8-sig")
print()
print(dist(ativ.origem, "treinos"))
print("contas por origem:", ativ.groupby("origem").user_id.nunique().to_dict())

treinos: 1252488 | contas com treino em 2026: 36551 de 63929 (57.2%)


           treinos  contas  duracao_mediana_min  % dos treinos  % das contas que treinam  treinos por conta
tipo                                                                                                       
corrida     385303   19188                 39.5           30.8                      52.5               20.1
funcional   254989   26839                 32.5           20.4                      73.4                9.5
caminhada   206193   23274                 58.6           16.5                      63.7                8.9
forca       197965   25235                 47.5           15.8                      69.0                7.8
yoga        113686   21584                 37.4            9.1                      59.1                5.3
pedal        94352    3618                 68.8            7.5                       9.9               26.1

              treinos    %
origem                    
app            879719 70.2
garmin         207954 16.6
apple_health   164815 13.2


contas por origem: {'app': 33747, 'apple_health': 6337, 'garmin': 7360}


In [13]:
pc = ativ.groupby(["user_id", "tipo"]).size().unstack(fill_value=0)
tot = pc.sum(axis=1)
conta = pd.DataFrame({"treinos": tot, "n_tipos": (pc > 0).sum(axis=1),
                      "principal": pc.idxmax(axis=1), "peso_principal": pc.max(axis=1) / tot,
                      "dias_ativos": ativ.groupby("user_id").data.nunique(),
                      "meses_ativos": ativ.groupby("user_id").data.apply(lambda s: s.dt.to_period("M").nunique())})
print("contas com treino:", len(conta))
print(); print("quantos tipos de treino cada conta pratica"); print(dist(conta.n_tipos).sort_index())
print(); print("esporte principal (o mais frequente da conta)"); print(dist(conta.principal))
print(); print("peso do esporte principal nos treinos da conta:", conta.peso_principal.describe()[["mean", "25%", "50%", "75%"]].round(2).to_dict())
print("treinos por conta em 2026:", conta.treinos.describe()[["mean", "25%", "50%", "75%", "max"]].round(1).to_dict())
print("meses com treino (de 9):", conta.meses_ativos.describe()[["mean", "25%", "50%", "75%"]].round(1).to_dict())

conta = conta.join(base.set_index("user_id")[["situacao", "ativo_", "tem_assinatura", "sexo_", "faixa_idade"]])
print(); print("contas com treino por situação"); print(dist(conta.situacao).sort_index())
print(); print("esporte principal × situação (% da coluna)")
conta["grupo"] = conta.situacao.str[3:].where(conta.tem_assinatura, "nunca assinou")
print(pd.crosstab(conta.principal, conta.grupo, normalize="columns", margins=True) * 100)
print(); print("esporte principal × sexo (% da coluna)")
print(pd.crosstab(conta.principal, conta.sexo_, normalize="columns") * 100)
print(); print("esporte principal × faixa de idade (% da coluna)")
print(pd.crosstab(conta.principal, conta.faixa_idade, normalize="columns") * 100)

contas com treino: 36551

quantos tipos de treino cada conta pratica
         contas    %
n_tipos             
1          5796 15.9
2          5870 16.1
3          5436 14.9
4         11351 31.1
5          8098 22.2

esporte principal (o mais frequente da conta)
           contas    %
principal             
corrida     14226 38.9
funcional    8259 22.6
caminhada    5503 15.1
forca        4401 12.0
pedal        3459  9.5
yoga          703  1.9

peso do esporte principal nos treinos da conta: {'mean': 0.66, '25%': 0.5, '50%': 0.67, '75%': 0.79}
treinos por conta em 2026: {'mean': 34.3, '25%': 4.0, '50%': 13.0, '75%': 52.0, 'max': 476.0}
meses com treino (de 9): {'mean': 4.1, '25%': 2.0, '50%': 3.0, '75%': 6.0}

contas com treino por situação
                               contas    %
situacao                                  
1. assinante ativo              14577 39.9
2. ex-assinante                  7985 21.8
3. em teste grátis                331  0.9
4. teste expirou, não assinou    85

faixa_idade  25-34  35-44  45-54  55+  até 24  não informado
principal                                                   
caminhada     15.3   14.7   15.5 15.1    13.2           14.2
corrida       39.0   39.0   39.1 38.7    34.9           38.4
forca         12.1   12.5   11.6 11.7    13.8           12.8
funcional     21.6   22.1   22.7 23.5    22.2           23.1
pedal          9.5    9.8    9.2  9.1    12.7           10.2
yoga           2.5    1.8    1.9  1.9     3.2            1.3


In [14]:
# Só assinantes ativos, último mês antes do corte
set_ = ativ[(ativ.data >= "2026-09-01") & ativ.user_id.isin(base.loc[base.ativo_, "user_id"])]
print("treinos de assinantes ativos em set/2026:", len(set_), "| contas:", set_.user_id.nunique())
t = set_.groupby("tipo").agg(treinos=("atividade_id", "size"), contas=("user_id", "nunique"))
t["% dos treinos"] = t.treinos / t.treinos.sum() * 100
t["% dos ativos que treinaram"] = t.contas / set_.user_id.nunique() * 100
print(t.sort_values("treinos", ascending=False))
print()
print("treinos por mês (todas as contas) e contas que treinaram")
m = ativ.assign(mes=ativ.data.dt.to_period("M")).groupby("mes").agg(treinos=("atividade_id", "size"), contas=("user_id", "nunique"))
m["treinos por conta"] = m.treinos / m.contas
print(m)

treinos de assinantes ativos em set/2026: 140185 | contas: 13055
           treinos  contas  % dos treinos  % dos ativos que treinaram
tipo                                                                 
corrida      41222    6289           29.4                        48.2
funcional    29475    9246           21.0                        70.8
caminhada    23280    7787           16.6                        59.6
forca        22665    8332           16.2                        63.8
yoga         13116    6562            9.4                        50.3
pedal        10427    1259            7.4                         9.6

treinos por mês (todas as contas) e contas que treinaram


         treinos  contas  treinos por conta
mes                                        
2026-01   157458   16999                9.3
2026-02   121009   16583                7.3
2026-03   139246   17298                8.0
2026-04   143092   17081                8.4
2026-05   145759   16725                8.7
2026-06   126339   16006                7.9
2026-07   125871   15952                7.9
2026-08   142780   16603                8.6
2026-09   150934   17325                8.7


### 5.2 Uso como rede social

Três sinais: seguir e ser seguido, postar no feed e treinar acompanhado (`treino_com`).

In [15]:
base = base.merge(rede.drop(columns=["assinante_ativo_no_corte"]), on="user_id", how="left", validate="one_to_one")
pp = posts.groupby("user_id").agg(posts=("post_id", "size"), ultimo_post=("data", "max"))
p90 = posts[posts.data > CORTE - pd.Timedelta(days=90)].groupby("user_id").size().rename("posts_90d")
tc = ativ[ativ.treino_com.notna()].groupby("user_id").size().rename("treinos_acompanhados")
base = base.merge(pp, on="user_id", how="left").merge(p90, on="user_id", how="left").merge(tc, on="user_id", how="left")
for c in ["posts", "posts_90d", "treinos_acompanhados"]:
    base[c] = base[c].fillna(0).astype(int)
print("contas depois das junções:", len(base))
print("posts:", len(posts), "de", posts.data.min().date(), "a", posts.data.max().date(), "| contas que postaram:", posts.user_id.nunique())
print("treinos acompanhados:", ativ.treino_com.notna().sum(), f"({ativ.treino_com.notna().mean():.1%} dos treinos) | contas:", ativ.loc[ativ.treino_com.notna(), 'user_id'].nunique())

base["tem_rede"] = (base.n_seguidores + base.n_seguindo) > 0
base["tem_mutua"] = base.n_mutuas > 0
base["postou"] = base.posts > 0
base["postou_90d"] = base.posts_90d > 0
base["treinou_acompanhado"] = base.treinos_acompanhados > 0
base["grupo"] = base.situacao.str[3:].where(base.tem_assinatura, "nunca assinou")
sinais = ["tem_rede", "tem_mutua", "postou", "postou_90d", "treinou_acompanhado"]
t = base.groupby("grupo")[sinais].mean().T * 100
t["todas as contas"] = base[sinais].mean() * 100
print(); print("% das contas com cada sinal social"); print(t)
print(); print("contas por grupo:", base.grupo.value_counts().to_dict())

contas depois das junções: 63929
posts: 119187 de 2025-07-01 a 2026-09-30 | contas que postaram: 26878
treinos acompanhados: 14112 (1.1% dos treinos) | contas: 2765

% das contas com cada sinal social
grupo                assinante ativo  ex-assinante  nunca assinou  todas as contas
tem_rede                        81.1          90.1           83.4             84.7
tem_mutua                       65.8          76.0           68.3             69.9
postou                          90.0          58.7           10.0             42.0
postou_90d                      73.2           5.9            2.2             19.6
treinou_acompanhado             11.1           3.1            1.8              4.3

contas por grupo: {'nunca assinou': 31346, 'ex-assinante': 17830, 'assinante ativo': 14753}


In [16]:
a = base[base.ativo_]
print("assinantes ativos:", len(a))
for c in ["n_seguidores", "n_seguindo", "n_mutuas", "n_seguindo_assinantes_ativos"]:
    print(f"{c:30s}", a[c].describe()[["mean", "25%", "50%", "75%", "max"]].round(1).to_dict())
print()
print("posts por conta que postou (todas):", base.loc[base.postou, "posts"].describe()[["mean", "25%", "50%", "75%", "max"]].round(1).to_dict())
print()
# Perfis de uso entre os ativos: treina? usa a rede (postou nos últimos 90 dias)?
a = a.assign(treina=(a.treinos_30d > 0).map({True: "treinou em 30d", False: "não treinou em 30d"}),
             social=a.postou_90d.map({True: "postou em 90d", False: "não postou em 90d"}))
print("ativos: treino × post (contas e % do total)")
print(pd.crosstab(a.treina, a.social, margins=True))
print(pd.crosstab(a.treina, a.social, normalize="all", margins=True) * 100)
print()
print("posts por mês e contas que postaram")
m = posts.assign(mes=posts.data.dt.to_period("M")).groupby("mes").agg(posts=("post_id", "size"), contas=("user_id", "nunique"))
print(m.tail(15))
print()
print("ativos: % com cada sinal social por plano")
print(a.groupby("plano_atual")[sinais].mean().T * 100)
print()
print("contas com treino: % que postou, por esporte principal")
c2 = conta.join(base.set_index("user_id")[["postou", "tem_rede", "treinou_acompanhado"]])
print(c2.groupby("principal")[["postou", "tem_rede", "treinou_acompanhado"]].mean() * 100)

base.groupby("situacao").agg(contas=("user_id", "size"), treinou_30d=("treinos_30d", lambda s: (s > 0).mean() * 100),
                             tem_rede=("tem_rede", lambda s: s.mean() * 100), postou=("postou", lambda s: s.mean() * 100),
                             postou_90d=("postou_90d", lambda s: s.mean() * 100),
                             treinou_acompanhado=("treinou_acompanhado", lambda s: s.mean() * 100)).to_csv(S / "eda_visao_geral_situacao.csv", encoding="utf-8-sig")

assinantes ativos: 14753
n_seguidores                   {'mean': 1.7, '25%': 0.0, '50%': 1.0, '75%': 2.0, 'max': 13.0}
n_seguindo                     {'mean': 1.7, '25%': 1.0, '50%': 1.0, '75%': 2.0, 'max': 11.0}
n_mutuas                       {'mean': 1.2, '25%': 0.0, '50%': 1.0, '75%': 2.0, 'max': 10.0}
n_seguindo_assinantes_ativos   {'mean': 0.4, '25%': 0.0, '50%': 0.0, '75%': 1.0, 'max': 6.0}

posts por conta que postou (todas): {'mean': 4.4, '25%': 1.0, '50%': 3.0, '75%': 6.0, 'max': 28.0}

ativos: treino × post (contas e % do total)


social              não postou em 90d  postou em 90d    All
treina                                                     
não treinou em 30d                817            881   1698
treinou em 30d                   3143           9912  13055
All                              3960          10793  14753
social              não postou em 90d  postou em 90d   All
treina                                                    
não treinou em 30d                5.5            6.0  11.5
treinou em 30d                   21.3           67.2  88.5
All                              26.8           73.2 100.0

posts por mês e contas que postaram
         posts  contas
mes                   
2025-07   6804    5179
2025-08   7319    5505
2025-09   7356    5573
2025-10   7832    5926
2025-11   7664    5814
2025-12   8070    6013
2026-01   8832    6577
2026-02   7317    5769
2026-03   8347    6367
2026-04   8058    6210
2026-05   8733    6569
2026-06   7851    6103
2026-07   7893    6120
2026-08   8359    6421


           postou  tem_rede  treinou_acompanhado
principal                                       
caminhada    51.6      79.8                  6.7
corrida      64.6      81.1                  8.0
forca        55.0      79.3                  6.2
funcional    70.7      82.6                  8.7
pedal        65.9      81.7                  7.3
yoga         37.3      77.7                  2.4
